# Extract training samples
* Download low-tide cloud free satellite iamges closest to the UAV image collection
* Sample the satellite image bands where appromximately a single UAV class

In [1]:
import pathlib
import numpy
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:60293/status,
Dashboard: http://127.0.0.1:60293/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:60294,Workers: 0
Dashboard: http://127.0.0.1:60293/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:60329,Total threads: 2
Dashboard: http://127.0.0.1:60330/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:60301,


2026-10-27 09:22:02,510 - distributed.scheduler - WARNING - Worker failed to heartbeat for 2081s; attempting restart: <WorkerState 'tcp://127.0.0.1:60318', name: 3, status: running, memory: 0, processing: 0>
2026-10-27 09:22:08,316 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing


# Values to edit

In [8]:
max_cloud_covers = [15] #[1, 5, 10] # percentage
tide_deltas = {1: {"hrs": 1, "mins": 00}, 
               2: {"hrs": 0, "mins": 45}, 
               3: {"hrs": 0, "mins": 30}}
lowtide_search_range = 60 # days - +/- this devided by 2

In [9]:
site_names = ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

# Cells to run

In [10]:
data_path = utils.get_data_path()
utils.create_data_folders()

training_labels_file = data_path / "ELF24505_ClassificationClasses.txt"
servey_dates_file = data_path / "ELF24505_SurveyDates.csv" 
uav_folder = data_path / "classified_uav"

In [11]:
all_baselines = sentinel2.load_band_quality_baseline()

for tide_delta in tide_deltas.values():
    for max_cloud_cover in max_cloud_covers:
        print(f"\n\nMax cloud percentage {max_cloud_cover}, Low tide delta {tide_delta['hrs']}hrs and {tide_delta['mins']}mins")
        
        for site_name in site_names:
            site_baseline = all_baselines.loc[all_baselines["site"] == site_name]
            for i in range(3):
                try:
                    sampling.get_site_satellite(
                        site_name=site_name,
                        survey_dates_file=servey_dates_file,
                        lowtide_search_range=lowtide_search_range,
                        low_tide_delta_hrs=tide_delta["hrs"],
                        low_tide_delta_mins=tide_delta["mins"],
                        uav_folder=uav_folder,
                        max_cloud_cover=max_cloud_cover,
                        anomaly_filter_baseline=site_baseline,
                        n_mad=3,
                    )
                    break
                except Exception:
                    print("Caught exception downloading images and try again.")
            print(f"Save RGB for {site_name}")
            satellite_images_path = utils.get_training_satellite_path(site_name=site_name,  low_tide_delta_hrs=tide_delta["hrs"],
                                                             low_tide_delta_mins=tide_delta["mins"], max_cloud_cover=max_cloud_cover,
                                                             filtered=True)
            utils.rgb_from_satellite(satellite_data_path=satellite_images_path)



Max cloud percentage 15, Low tide delta 1hrs and 0mins
Site Ihutai
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2025-03-23/2025-05-22
	Tide API-filtered tiles: 4 from 13 STAC items
	Tide API-filtered tiles: 4 from 13 STAC items
	Low Cloud Tiles: 2 from the low tide tiles of 4. Cloud percentages [ 21.8   0.   12.7 100. ] Kept [False  True  True False]
		2025-04-12: B04=0.06, B03=0.79, B02=1.29 -> kept
		2025-04-27: B04=3.41, B03=4.15, B02=3.52 -> dropped
	Anomaly filter: 1 of 2 dates kept for site Ihutai (n_mad=3)
Save RGB for Ihutai
Site CatlinsLake
	Save satellite image of the site around lowtide without cloud
	Satellite date range 2022-07-11/2022-09-09
	Tide API-filtered tiles: 5 from 34 STAC items
	Tide API-filtered tiles: 4 from 34 STAC items
	Low Cloud Tiles: 1 from the low tide tiles of 4. Cloud percentages [  0.1 100.   88.  100. ] Kept [ True False False False]
		2022-07-15: B04=1.20, B03=0.21, B02=1.21 -> kept
	Anomaly filter: 1 of 1 d

2026-10-27 09:22:13,480 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
